# Module 09: Copilot Agent API Testing Harness

## Learning Objectives

By completing this notebook, you will:
1. Authenticate against the Power Platform REST API to list Copilot Studio agents
2. Retrieve agent configuration details programmatically
3. Send conversational messages to an agent via the Direct Line API
4. Parse conversation transcripts and validate expected responses
5. Build a reusable test harness for automated agent regression testing

## Prerequisites

- Azure AD app registration with Power Platform API permissions
- A published Copilot Studio agent in your environment
- Python 3.9 or higher
- `requests` and `python-dotenv` libraries installed

## Estimated Time: 15 minutes

---

## Why Test Agents Programmatically?

Copilot Studio's built-in test canvas is valuable for manual exploration but has limitations:
- It does not scale across multiple scenarios
- It cannot run as part of a CI/CD pipeline
- It does not produce machine-readable pass/fail results

The **Direct Line API** (part of Azure Bot Service) provides a REST interface to send messages to any Copilot Studio agent and read its responses. This notebook builds a Python test harness on top of that API.

In [ ]:
learning_objectives([
    "Azure AD app registration with Power Platform API permissions",
    "A published Copilot Studio agent in your environment",
    "Python 3.9 or higher",
    "`requests` and `python-dotenv` libraries installed",
    "## Why Test Agents Programmatically?",
    "It does not scale across multiple scenarios",
    "It cannot run as part of a CI/CD pipeline",
    "It does not produce machine-readable pass/fail results",
    "(part of Azure Bot Service) provides a REST interface to send messages to any Copilot Studio agent and read its responses. This notebook builds a Python test harness on top of that API."
])

## 1. Setup and Configuration

We need two sets of credentials:
- **Azure AD token** — to authenticate against the Power Platform management APIs
- **Direct Line secret** — to open a conversation channel with the specific agent

Store these in a `.env` file in your project directory:

```
AZURE_TENANT_ID=your-tenant-id
AZURE_CLIENT_ID=your-app-registration-client-id
AZURE_CLIENT_SECRET=your-app-registration-secret
POWER_PLATFORM_ENVIRONMENT_ID=your-environment-id
DIRECT_LINE_SECRET=your-direct-line-secret-from-copilot-studio
```

### How to find your Direct Line secret

1. In Copilot Studio, open your agent
2. Click **Settings** → **Channels** → **Mobile app**
3. Copy the **Token Endpoint** URL and the **Secret** value shown on the page

### How to find your Environment ID

In the Power Platform admin center (admin.powerplatform.microsoft.com), open your environment's detail page. The environment ID appears in the URL: `.../environments/{environment-id}/...`

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Install required libraries if not already present
# Uncomment the line below on first run
# !pip install requests python-dotenv

import os
import json
import time
from typing import Optional
from dotenv import load_dotenv

try:
    import requests
except ImportError:
    raise ImportError(
        "The 'requests' library is required. Run: pip install requests"
    )

# Load environment variables from .env file
load_dotenv()

# Read configuration — all values come from environment variables, never hardcoded
TENANT_ID = os.getenv("AZURE_TENANT_ID", "")
CLIENT_ID = os.getenv("AZURE_CLIENT_ID", "")
CLIENT_SECRET = os.getenv("AZURE_CLIENT_SECRET", "")
ENVIRONMENT_ID = os.getenv("POWER_PLATFORM_ENVIRONMENT_ID", "")
DIRECT_LINE_SECRET = os.getenv("DIRECT_LINE_SECRET", "")

# Validate that required variables are present
required_vars = {
    "AZURE_TENANT_ID": TENANT_ID,
    "AZURE_CLIENT_ID": CLIENT_ID,
    "AZURE_CLIENT_SECRET": CLIENT_SECRET,
    "POWER_PLATFORM_ENVIRONMENT_ID": ENVIRONMENT_ID,
    "DIRECT_LINE_SECRET": DIRECT_LINE_SECRET,
}

missing = [name for name, value in required_vars.items() if not value]
if missing:
    print(f"[WARNING] Missing environment variables: {', '.join(missing)}")
    print("The notebook will demonstrate API patterns but cannot make live calls.")
    print("Set the missing values in your .env file to run against a real environment.")
else:
    print("[OK] All required environment variables are set.")

## 2. Authenticating with Azure AD

Power Platform APIs use OAuth 2.0 with client credentials flow. We request a bearer token from the Azure AD token endpoint, then include it in the `Authorization` header of every API call.

The **resource** (audience) for Power Platform management APIs is:
```
https://service.powerapps.com/
```

The bearer token is valid for one hour. The `get_access_token()` function below requests a fresh token each time — in production you would cache and refresh it.

In [ ]:
# Purpose: Obtain an Azure AD bearer token via the client credentials (app-only) flow.
# The token is used in the Authorization header for all Power Platform API calls.

def get_access_token(
    tenant_id: str,
    client_id: str,
    client_secret: str,
    resource: str = "https://service.powerapps.com/",
) -> str:
    """
    Retrieve an OAuth 2.0 access token from Azure AD using client credentials.

    Parameters
    ----------
    tenant_id : str
        The Azure AD tenant ID (GUID).
    client_id : str
        The app registration client ID (GUID).
    client_secret : str
        The app registration client secret value.
    resource : str
        The resource (audience) URI. Default is Power Apps service.

    Returns
    -------
    str
        The access token string (without the 'Bearer ' prefix).

    Raises
    ------
    requests.HTTPError
        If the token request fails (e.g., invalid credentials).
    """
    token_url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"

    # Client credentials grant type — no user interaction required
    payload = {
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
        "resource": resource,
    }

    response = requests.post(token_url, data=payload, timeout=30)
    response.raise_for_status()

    token_data = response.json()
    return token_data["access_token"]


# Attempt to get a token — skip gracefully if credentials are not configured
access_token = None

if TENANT_ID and CLIENT_ID and CLIENT_SECRET:
    try:
        access_token = get_access_token(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
        print(f"[OK] Access token obtained. Length: {len(access_token)} characters.")
        print(f"     Token preview: {access_token[:40]}...")
    except requests.HTTPError as exc:
        print(f"[ERROR] Token request failed: {exc}")
else:
    print("[SKIP] Credentials not configured — skipping live token request.")
    print("       Demonstrating the API structure only.")

## 3. Listing Agents in a Power Platform Environment

The Power Platform API exposes a `/bots` endpoint that returns all Copilot Studio agents (bots) in a given environment. This is useful for:
- Discovering which agents exist without logging into Copilot Studio
- Retrieving the agent's `botId` for subsequent API calls
- Monitoring deployment state across environments

**API endpoint:**
```
GET https://api.powerplatform.com/powervirtualagents/environments/{environmentId}/bots
    ?api-version=2022-03-01-preview
```

**Required permission:** The app registration must have the `Dynamics CRM user_impersonation` API permission or the `Power Platform API` permission depending on the tenant configuration.

In [ ]:
# Purpose: Query the Power Platform API to list all agents in the configured environment.
# This lets you inspect what agents are deployed without opening Copilot Studio.

def list_agents(
    environment_id: str,
    access_token: str,
    api_version: str = "2022-03-01-preview",
) -> list[dict]:
    """
    Retrieve all Copilot Studio agents in the specified Power Platform environment.

    Parameters
    ----------
    environment_id : str
        The Power Platform environment ID (GUID).
    access_token : str
        Bearer token from get_access_token().
    api_version : str
        API version string for the Power Platform API.

    Returns
    -------
    list[dict]
        List of agent objects. Each contains: id, displayName, schemaName,
        publishedBotStatus, createdOn, modifiedOn.
    """
    base_url = "https://api.powerplatform.com"
    endpoint = f"{base_url}/powervirtualagents/environments/{environment_id}/bots"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }

    params = {"api-version": api_version}

    response = requests.get(endpoint, headers=headers, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()
    # The API returns a paged response — for simplicity we return the first page
    return data.get("value", [])


# Run the listing call if we have both a token and an environment ID
agents = []

if access_token and ENVIRONMENT_ID:
    try:
        agents = list_agents(ENVIRONMENT_ID, access_token)
        print(f"[OK] Found {len(agents)} agent(s) in environment {ENVIRONMENT_ID[:8]}...")
        for agent in agents:
            status = agent.get("publishedBotStatus", "Unknown")
            name = agent.get("displayName", "Unnamed")
            agent_id = agent.get("id", "")
            print(f"     [{status:12}]  {name}  (id: {agent_id[:8]}...)")
    except requests.HTTPError as exc:
        print(f"[ERROR] Could not list agents: {exc}")
        if exc.response is not None:
            print(f"        Response body: {exc.response.text[:400]}")
else:
    print("[SKIP] No token or environment ID — showing example response structure instead.")
    # Demonstrate the expected response shape with sample data
    agents = [
        {
            "id": "a1b2c3d4-e5f6-7890-abcd-ef1234567890",
            "displayName": "IT Helpdesk Assistant",
            "schemaName": "new_ithelpdesk",
            "publishedBotStatus": "Published",
            "createdOn": "2026-03-01T10:00:00Z",
            "modifiedOn": "2026-03-08T14:30:00Z",
        }
    ]
    print(f"[DEMO] Example response with {len(agents)} agent(s):")
    for agent in agents:
        print(f"       {agent['displayName']} — status: {agent['publishedBotStatus']}")

## 4. Inspecting Agent Configuration

Once you have an agent ID, you can retrieve the detailed configuration — including the agent's topic count, language settings, and Direct Line token endpoint — via the individual agent endpoint.

This is useful for:
- Verifying the correct agent version is deployed
- Extracting the token endpoint URL without opening the Copilot Studio UI
- Checking publication status programmatically

In [ ]:
# Purpose: Retrieve detailed configuration for a specific Copilot Studio agent.
# The configuration object contains metadata needed for Direct Line API calls.

def get_agent_details(
    environment_id: str,
    bot_id: str,
    access_token: str,
    api_version: str = "2022-03-01-preview",
) -> dict:
    """
    Retrieve detailed configuration for a specific Copilot Studio agent.

    Parameters
    ----------
    environment_id : str
        The Power Platform environment ID.
    bot_id : str
        The agent (bot) ID from list_agents().
    access_token : str
        Bearer token from get_access_token().
    api_version : str
        API version string.

    Returns
    -------
    dict
        Agent detail object including: displayName, publishedBotStatus,
        cdsBotId, tokenEndpointUrl, defaultLanguage, topics (count).
    """
    base_url = "https://api.powerplatform.com"
    endpoint = (
        f"{base_url}/powervirtualagents/environments/{environment_id}/bots/{bot_id}"
    )

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Accept": "application/json",
    }

    response = requests.get(
        endpoint, headers=headers, params={"api-version": api_version}, timeout=30
    )
    response.raise_for_status()
    return response.json()


# Retrieve details for the first agent found (or use example data)
agent_details = None

if agents and access_token and ENVIRONMENT_ID:
    first_agent_id = agents[0]["id"]
    try:
        agent_details = get_agent_details(ENVIRONMENT_ID, first_agent_id, access_token)
        print("[OK] Agent details retrieved:")
        print(f"     Display name:   {agent_details.get('displayName')}")
        print(f"     Status:         {agent_details.get('publishedBotStatus')}")
        print(f"     Language:       {agent_details.get('defaultLanguage', 'en-US')}")
        print(f"     Token endpoint: {agent_details.get('tokenEndpointUrl', 'N/A')[:60]}...")
    except requests.HTTPError as exc:
        print(f"[ERROR] Could not retrieve agent details: {exc}")
else:
    # Demonstrate expected structure with example data
    agent_details = {
        "id": "a1b2c3d4-e5f6-7890-abcd-ef1234567890",
        "displayName": "IT Helpdesk Assistant",
        "publishedBotStatus": "Published",
        "defaultLanguage": "en-US",
        "tokenEndpointUrl": "https://directline.botframework.com/v3/directline",
        "cdsBotId": "new_ithelpdesk",
    }
    print("[DEMO] Example agent details structure:")
    for key, value in agent_details.items():
        print(f"       {key}: {value}")

## 5. Direct Line API: Opening a Conversation

The **Direct Line API** is the standard protocol for interacting with any Azure Bot Service-based bot (including Copilot Studio agents) from code. It provides three operations:

1. **Start conversation** — obtain a `conversationId` and `token`
2. **Send activity** — post a message to the bot within a conversation
3. **Get activities** — retrieve the bot's responses from the conversation stream

**Base URL:** `https://directline.botframework.com/v3/directline`

**Authentication:** Pass the Direct Line secret in the `Authorization` header as `Bearer {secret}`, or first exchange the secret for a per-conversation token at `/tokens/generate`.

In [ ]:
# Purpose: Manage a Direct Line conversation session with a Copilot Studio agent.
# The DirectLineSession class encapsulates the three Direct Line operations:
# start_conversation, send_message, and get_responses.

DIRECT_LINE_BASE = "https://directline.botframework.com/v3/directline"


class DirectLineSession:
    """
    Manages a conversation session with a Copilot Studio agent via Direct Line API.

    Usage
    -----
    session = DirectLineSession(direct_line_secret)
    session.start()
    session.send("search knowledge base")
    responses = session.receive(timeout_seconds=10)
    """

    def __init__(self, secret: str) -> None:
        """
        Parameters
        ----------
        secret : str
            Direct Line secret from Copilot Studio → Settings → Channels → Mobile app.
        """
        self.secret = secret
        self.conversation_id: Optional[str] = None
        self.token: Optional[str] = None
        # Watermark tracks which activities have already been read
        # so we do not re-read earlier messages on subsequent calls
        self._watermark: Optional[str] = None

    def _headers(self) -> dict:
        """Return Authorization headers using the current token or secret."""
        auth_value = self.token if self.token else self.secret
        return {
            "Authorization": f"Bearer {auth_value}",
            "Content-Type": "application/json",
        }

    def start(self) -> str:
        """
        Open a new conversation and store the conversation ID.

        Returns
        -------
        str
            The conversation ID for this session.
        """
        url = f"{DIRECT_LINE_BASE}/conversations"
        response = requests.post(url, headers=self._headers(), timeout=30)
        response.raise_for_status()

        data = response.json()
        # Store per-conversation token (more secure than reusing the master secret)
        self.token = data.get("token", self.secret)
        self.conversation_id = data["conversationId"]
        return self.conversation_id

    def send(self, text: str, from_id: str = "test-user") -> None:
        """
        Send a message activity to the agent.

        Parameters
        ----------
        text : str
            The user's message text.
        from_id : str
            Identifier for the sending user. Used in activity routing.
        """
        if not self.conversation_id:
            raise RuntimeError("Call start() before send()")

        url = f"{DIRECT_LINE_BASE}/conversations/{self.conversation_id}/activities"

        # Activity schema follows the Bot Framework Activity specification
        activity = {
            "type": "message",
            "from": {"id": from_id},
            "text": text,
        }

        response = requests.post(
            url, headers=self._headers(), json=activity, timeout=30
        )
        response.raise_for_status()

    def receive(self, timeout_seconds: int = 15, poll_interval: float = 1.0) -> list[dict]:
        """
        Poll for new activities from the agent until the timeout elapses.

        Parameters
        ----------
        timeout_seconds : int
            Maximum time to wait for a response before giving up.
        poll_interval : float
            Seconds between poll requests.

        Returns
        -------
        list[dict]
            List of activity objects from the agent (type='message', from.id='bot').
        """
        if not self.conversation_id:
            raise RuntimeError("Call start() before receive()")

        url = f"{DIRECT_LINE_BASE}/conversations/{self.conversation_id}/activities"

        deadline = time.time() + timeout_seconds
        bot_messages = []

        while time.time() < deadline:
            params = {}
            if self._watermark:
                # Only fetch activities after the last seen watermark
                params["watermark"] = self._watermark

            response = requests.get(
                url, headers=self._headers(), params=params, timeout=30
            )
            response.raise_for_status()

            data = response.json()
            self._watermark = data.get("watermark", self._watermark)

            # Filter to bot-originated message activities only
            new_messages = [
                act
                for act in data.get("activities", [])
                if act.get("type") == "message"
                and act.get("from", {}).get("role") == "bot"
            ]

            if new_messages:
                bot_messages.extend(new_messages)
                break  # Stop polling once we have at least one response

            time.sleep(poll_interval)

        return bot_messages


print("[OK] DirectLineSession class defined.")
print("     Usage: session = DirectLineSession(DIRECT_LINE_SECRET)")
print("            session.start()")
print("            session.send('how do I reset my password')")
print("            responses = session.receive()")

## 6. Sending a Message and Reading the Response

With `DirectLineSession` defined, we can now conduct a full conversation turn:
1. Open a conversation
2. Send a user message
3. Poll for and display the agent's response

The `text` field of each activity contains the agent's reply. Additional fields like `attachments` and `suggestedActions` contain structured data and quick-reply buttons respectively.

In [ ]:
# Purpose: Demonstrate a complete conversation turn using DirectLineSession.
# This is the minimal working pattern used in all test cases below.

def send_and_receive(
    session: DirectLineSession,
    message: str,
    timeout_seconds: int = 15,
) -> list[str]:
    """
    Send a message to the agent and return its text responses.

    Parameters
    ----------
    session : DirectLineSession
        An active (started) session.
    message : str
        The user message to send.
    timeout_seconds : int
        How long to wait for a response before giving up.

    Returns
    -------
    list[str]
        Text content of all bot message activities received in response.
    """
    session.send(message)
    activities = session.receive(timeout_seconds=timeout_seconds)
    # Extract text from each message activity; fall back to empty string
    return [act.get("text", "") for act in activities]


# Demonstrate the pattern (live call requires DIRECT_LINE_SECRET to be set)
if DIRECT_LINE_SECRET:
    try:
        session = DirectLineSession(DIRECT_LINE_SECRET)
        conv_id = session.start()
        print(f"[OK] Conversation started. ID: {conv_id}")

        # First turn: trigger the greeting topic
        greeting_responses = send_and_receive(session, "Hello")
        print("\n[Agent response to 'Hello':]")
        for text in greeting_responses:
            print(f"     {text}")

    except requests.HTTPError as exc:
        print(f"[ERROR] Direct Line call failed: {exc}")
        if exc.response is not None:
            print(f"        Status: {exc.response.status_code}")
            print(f"        Body: {exc.response.text[:300]}")
else:
    print("[SKIP] DIRECT_LINE_SECRET not set — showing expected output structure.")
    print()
    print("Expected output when connected to IT Helpdesk Assistant:")
    print("[OK] Conversation started. ID: AbCd1234efGh...")
    print()
    print("[Agent response to 'Hello':]")
    print("     Hello! I'm the IT Helpdesk Assistant. I can help you search for")
    print("     IT knowledge base articles, create and track support tickets, or")
    print("     escalate critical issues. How can I help you today?")

## 7. Building the Test Harness

A test harness runs a predefined set of message inputs and validates the agent's responses against expected criteria. For a Copilot agent, "validation" is typically:

- **Topic routing check** — did the response contain topic-specific content?
- **Key phrase check** — does the response include expected keywords or phrases?
- **No-error check** — the agent did not respond with its fallback error topic

We represent each test case as a dictionary and the harness runs them in sequence.

In [ ]:
# Purpose: Define test cases and a runner that validates agent responses.
# Each test case specifies a user message and the criteria the response must meet.

from dataclasses import dataclass, field


@dataclass
class AgentTestCase:
    """
    Specification for a single agent test scenario.

    Attributes
    ----------
    name : str
        Human-readable name for the test (used in reports).
    messages : list[str]
        Sequence of user messages to send in order (multi-turn conversation).
    expected_phrases : list[str]
        Phrases that must appear somewhere in the combined agent response text.
    forbidden_phrases : list[str]
        Phrases that must NOT appear (e.g., error messages, wrong topic names).
    """
    name: str
    messages: list[str]
    expected_phrases: list[str] = field(default_factory=list)
    forbidden_phrases: list[str] = field(default_factory=list)


# IT Helpdesk Agent test suite
# Each case tests one topic's basic routing and response content
IT_HELPDESK_TEST_CASES = [
    AgentTestCase(
        name="Greeting response",
        messages=["Hello"],
        expected_phrases=["helpdesk", "help"],
        forbidden_phrases=["I'm not sure", "Sorry, I didn't understand"],
    ),
    AgentTestCase(
        name="Search KB topic routing",
        messages=["search knowledge base"],
        expected_phrases=["looking for", "search", "topic"],
        forbidden_phrases=["Sorry, I didn't understand", "couldn't understand"],
    ),
    AgentTestCase(
        name="Create ticket topic routing",
        messages=["I need to submit a ticket"],
        expected_phrases=["category", "Hardware", "Software"],
        forbidden_phrases=["Sorry, I didn't understand", "I don't know"],
    ),
    AgentTestCase(
        name="Check ticket status topic routing",
        messages=["check my ticket status"],
        expected_phrases=["ticket ID", "INC-", "enter"],
        forbidden_phrases=["Sorry, I didn't understand"],
    ),
    AgentTestCase(
        name="Escalation topic routing",
        messages=["I need to escalate"],
        expected_phrases=["escalat", "manager", "review"],
        forbidden_phrases=["Sorry, I didn't understand"],
    ),
    AgentTestCase(
        name="Out-of-scope request handling",
        messages=["What is the weather today"],
        # Agent should either redirect or acknowledge it handles IT only
        # We do NOT expect a helpful weather answer
        expected_phrases=[],
        forbidden_phrases=[],  # Relaxed — just verify no crash
    ),
]

print(f"[OK] Test suite defined with {len(IT_HELPDESK_TEST_CASES)} test cases:")
for tc in IT_HELPDESK_TEST_CASES:
    print(f"     - {tc.name}")

In [ ]:
# Purpose: Execute the test suite against a live agent and report results.
# Each test case opens a fresh conversation to avoid state leakage between cases.

def run_test_suite(
    test_cases: list[AgentTestCase],
    direct_line_secret: str,
    response_timeout: int = 15,
) -> dict:
    """
    Run all test cases against a Copilot Studio agent via Direct Line.

    Parameters
    ----------
    test_cases : list[AgentTestCase]
        Test cases to execute.
    direct_line_secret : str
        Direct Line secret for the agent under test.
    response_timeout : int
        Seconds to wait per message before failing the test.

    Returns
    -------
    dict
        Summary: {passed: int, failed: int, errors: int, details: list[dict]}
    """
    results = {"passed": 0, "failed": 0, "errors": 0, "details": []}

    for tc in test_cases:
        detail = {"name": tc.name, "status": None, "failures": [], "responses": []}

        try:
            # Fresh conversation for each test case — prevents topic state from bleeding across tests
            session = DirectLineSession(direct_line_secret)
            session.start()

            # Send each message in the sequence, collecting all responses
            combined_response_text = ""
            for message in tc.messages:
                responses = send_and_receive(session, message, timeout_seconds=response_timeout)
                combined_response_text += " ".join(responses).lower()
                detail["responses"].extend(responses)

            # Check expected phrases
            for phrase in tc.expected_phrases:
                if phrase.lower() not in combined_response_text:
                    detail["failures"].append(
                        f"Expected phrase not found: '{phrase}'"
                    )

            # Check forbidden phrases
            for phrase in tc.forbidden_phrases:
                if phrase.lower() in combined_response_text:
                    detail["failures"].append(
                        f"Forbidden phrase found: '{phrase}'"
                    )

            if detail["failures"]:
                detail["status"] = "FAIL"
                results["failed"] += 1
            else:
                detail["status"] = "PASS"
                results["passed"] += 1

        except Exception as exc:
            detail["status"] = "ERROR"
            detail["failures"].append(f"Exception: {exc}")
            results["errors"] += 1

        results["details"].append(detail)

    return results


# Run the suite if a Direct Line secret is configured
test_results = None

if DIRECT_LINE_SECRET:
    print("Running test suite against live agent...")
    print("(Each test case opens a fresh conversation — this may take 30-90 seconds)")
    print()
    test_results = run_test_suite(IT_HELPDESK_TEST_CASES, DIRECT_LINE_SECRET)
else:
    print("[SKIP] DIRECT_LINE_SECRET not set — cannot run live tests.")
    print("       Set the secret in .env to run the full suite against your agent.")

## 8. Parsing and Displaying Test Results

The `test_results` dictionary from `run_test_suite()` contains detailed pass/fail information for each test case. The display function below formats this into a readable report.

In a CI/CD context, you would check `test_results["failed"] == 0` as the pipeline pass condition.

In [ ]:
# Purpose: Format and display test results in a readable report.
# This format is similar to pytest output — pass/fail per test with failure details.

def print_test_report(results: dict) -> None:
    """
    Print a formatted test report from run_test_suite() output.

    Parameters
    ----------
    results : dict
        Output from run_test_suite().
    """
    total = results["passed"] + results["failed"] + results["errors"]
    print(f"{'='*60}")
    print(f"AGENT TEST REPORT — {total} test(s) run")
    print(f"{'='*60}")

    for detail in results["details"]:
        status_symbol = {"PASS": "PASS", "FAIL": "FAIL", "ERROR": "ERROR"}.get(
            detail["status"], "????"
        )
        print(f"  [{status_symbol}]  {detail['name']}")

        if detail["failures"]:
            for failure in detail["failures"]:
                print(f"         REASON: {failure}")

        if detail["responses"]:
            # Show first 120 characters of the first response for context
            first_response = detail["responses"][0][:120]
            print(f"         Agent:  \"{first_response}\"")

    print(f"{'='*60}")
    print(
        f"Results: {results['passed']} passed, "
        f"{results['failed']} failed, "
        f"{results['errors']} errors"
    )

    if results["failed"] == 0 and results["errors"] == 0:
        print("All tests passed.")
    else:
        print("Review failures above before publishing the agent.")


# Display results if the suite ran, otherwise show example output
if test_results:
    print_test_report(test_results)
else:
    # Show what the output looks like when all tests pass
    example_results = {
        "passed": 5,
        "failed": 1,
        "errors": 0,
        "details": [
            {"name": "Greeting response", "status": "PASS", "failures": [], "responses": ["Hello! I'm the IT Helpdesk Assistant."]},
            {"name": "Search KB topic routing", "status": "PASS", "failures": [], "responses": ["I can search our IT knowledge base for you. What topic are you looking for?"]},
            {"name": "Create ticket topic routing", "status": "PASS", "failures": [], "responses": ["I'll help you create a support ticket. What category best describes your issue?"]},
            {"name": "Check ticket status topic routing", "status": "PASS", "failures": [], "responses": ["Please enter your ticket ID (for example: INC-2094):"]},
            {"name": "Escalation topic routing", "status": "PASS", "failures": [], "responses": ["I can escalate your issue to an IT manager."]},
            {"name": "Out-of-scope request handling", "status": "FAIL", "failures": ["Forbidden phrase found: 'I don't know'"], "responses": ["I don't know the current weather. I'm an IT support assistant."]},
        ],
    }
    print("[DEMO] Example test report output:")
    print()
    print_test_report(example_results)

## 9. Parsing Conversation Transcripts

The Direct Line API returns full activity objects, not just message text. Activities contain:
- `type` — `message`, `typing`, `event`
- `text` — the message text
- `suggestedActions` — quick-reply buttons the agent offered
- `attachments` — cards (Adaptive Cards, Hero Cards) with structured content
- `timestamp` — UTC timestamp of the activity

Parsing `suggestedActions` is important for testing topics that offer choices, because the test harness needs to "click" a button by sending the button's `value` as the next user message.

In [ ]:
# Purpose: Extract structured data from Direct Line activity objects.
# These helpers make transcript analysis readable and support multi-turn tests
# that need to interact with quick-reply buttons.

def extract_quick_replies(activity: dict) -> list[str]:
    """
    Extract quick-reply option values from a bot message activity.

    Copilot Studio quick replies appear in the suggestedActions field.
    Each action has a 'title' (display label) and 'value' (what to send).

    Parameters
    ----------
    activity : dict
        A message activity from DirectLineSession.receive().

    Returns
    -------
    list[str]
        The value strings for each suggested action (these are the text
        to send back to continue the conversation via a button press).
    """
    suggested = activity.get("suggestedActions", {})
    actions = suggested.get("actions", [])
    return [action.get("value", action.get("title", "")) for action in actions]


def format_transcript(activities: list[dict], user_id: str = "test-user") -> str:
    """
    Format a list of Direct Line activities as a human-readable transcript.

    Parameters
    ----------
    activities : list[dict]
        Activities from a DirectLineSession conversation.
    user_id : str
        The user's from.id used when sending messages.

    Returns
    -------
    str
        Multi-line transcript string with User:/Agent: prefixes.
    """
    lines = []
    for activity in activities:
        if activity.get("type") != "message":
            continue

        from_id = activity.get("from", {}).get("id", "")
        role = "User" if from_id == user_id else "Agent"
        text = activity.get("text", "[no text]")
        timestamp = activity.get("timestamp", "")[:19]  # Trim to seconds

        lines.append(f"[{timestamp}] {role:5}: {text}")

        # Show quick replies if present
        quick_replies = extract_quick_replies(activity)
        if quick_replies:
            lines.append(f"             Choices: {' | '.join(quick_replies)}")

    return "\n".join(lines)


# Demonstrate with example transcript data
example_activities = [
    {
        "type": "message",
        "from": {"id": "test-user"},
        "text": "I need to submit a ticket",
        "timestamp": "2026-03-08T14:30:00.000Z",
    },
    {
        "type": "message",
        "from": {"id": "bot", "role": "bot"},
        "text": "I'll help you create a support ticket. What category best describes your issue?",
        "timestamp": "2026-03-08T14:30:01.500Z",
        "suggestedActions": {
            "actions": [
                {"type": "imBack", "title": "Hardware", "value": "Hardware"},
                {"type": "imBack", "title": "Software", "value": "Software"},
                {"type": "imBack", "title": "Network", "value": "Network"},
                {"type": "imBack", "title": "Account", "value": "Account"},
            ]
        },
    },
    {
        "type": "message",
        "from": {"id": "test-user"},
        "text": "Software",
        "timestamp": "2026-03-08T14:30:05.000Z",
    },
    {
        "type": "message",
        "from": {"id": "bot", "role": "bot"},
        "text": "Briefly describe the issue:",
        "timestamp": "2026-03-08T14:30:06.200Z",
    },
]

print("Example conversation transcript:")
print("-" * 60)
print(format_transcript(example_activities))
print()

# Show how to extract quick replies for use in the next turn
bot_turn = example_activities[1]
choices = extract_quick_replies(bot_turn)
print(f"Quick-reply options extracted from agent turn: {choices}")
print(f"To select 'Software' in a test: session.send(choices[1])")

## Exercises

### Exercise 9.1: Multi-Turn Test Case

**Task:** Write a test case for the Create Ticket topic that sends multiple messages (simulating a full conversation through all prompts) and validates that the final response contains a ticket ID starting with 'INC-'.

**Hints:**
- A test case can have more than one message in `messages` — the harness sends them in order
- The run_test_suite function waits for the agent to respond after each message before sending the next
- The final agent message should contain "INC-" if the Create Ticket flow ran successfully

In [ ]:
# YOUR CODE HERE
# ---------------
# Create an AgentTestCase for a full Create Ticket conversation.
# The messages list should drive the agent through: category → description → priority → confirm

create_ticket_multi_turn_test = None  # Replace with an AgentTestCase instance

# Example structure to guide you:
# create_ticket_multi_turn_test = AgentTestCase(
#     name="...",
#     messages=[...],          # All the messages to send in sequence
#     expected_phrases=[...],  # What must appear in the combined response
#     forbidden_phrases=[...], # What must NOT appear
# )

In [ ]:
# AUTO-GRADED TESTS — do not modify
# -----------------------------------

def test_exercise_9_1():
    assert create_ticket_multi_turn_test is not None, (
        "create_ticket_multi_turn_test must be defined — it should be an AgentTestCase instance"
    )
    assert isinstance(create_ticket_multi_turn_test, AgentTestCase), (
        "create_ticket_multi_turn_test must be an AgentTestCase instance"
    )
    assert len(create_ticket_multi_turn_test.messages) >= 3, (
        "A Create Ticket test needs at least 3 messages: trigger phrase, category, description. "
        f"Got {len(create_ticket_multi_turn_test.messages)}."
    )
    assert any(
        "ticket" in msg.lower() or "submit" in msg.lower() or "issue" in msg.lower()
        for msg in create_ticket_multi_turn_test.messages
    ), (
        "At least one message should trigger the Create Ticket topic "
        "(contain 'ticket', 'submit', or 'issue')."
    )
    assert any(
        "INC" in phrase for phrase in create_ticket_multi_turn_test.expected_phrases
    ), (
        "expected_phrases should include 'INC-' to validate the ticket ID was returned."
    )
    print("[PASS] Exercise 9.1 passed.")
    print(f"       Test name: '{create_ticket_multi_turn_test.name}'")
    print(f"       Messages:  {len(create_ticket_multi_turn_test.messages)} turns")
    print(f"       Expected:  {create_ticket_multi_turn_test.expected_phrases}")


test_exercise_9_1()

### Exercise 9.2: Response Validator Function

**Task:** Write a function `validate_ticket_id_in_response(responses: list[str]) -> bool` that returns `True` if any response in the list contains a string matching the pattern `INC-` followed by digits (e.g., `INC-2094`).

**Hints:**
- Use the `re` (regular expressions) module
- The pattern is: `INC-` followed by one or more digits: `r'INC-\d+'`
- Search across all responses, not just the first one

In [ ]:
# YOUR CODE HERE
# ---------------
import re

def validate_ticket_id_in_response(responses: list[str]) -> bool:
    """
    Check whether any response string contains a formatted ticket ID.

    Parameters
    ----------
    responses : list[str]
        Agent response text strings from a conversation turn.

    Returns
    -------
    bool
        True if any response contains a ticket ID matching 'INC-' + digits.
    """
    # TODO: Implement using re.search()
    pass

exercise_9_2_answer = validate_ticket_id_in_response

In [ ]:
# AUTO-GRADED TESTS — do not modify
# -----------------------------------

def test_exercise_9_2():
    assert exercise_9_2_answer is not None, "Implement validate_ticket_id_in_response"

    # Case 1: ticket ID present
    result = exercise_9_2_answer(["Your ticket INC-2094 has been created."])
    assert result is True, (
        "Expected True when 'INC-2094' is in the response. "
        f"Got: {result}"
    )

    # Case 2: no ticket ID
    result = exercise_9_2_answer(["What category best describes your issue?"])
    assert result is False, (
        "Expected False when no ticket ID is present. "
        f"Got: {result}"
    )

    # Case 3: ticket ID in second response of a multi-response list
    result = exercise_9_2_answer([
        "I'll help you create a support ticket.",
        "Ticket INC-107 has been created and assigned to the Software Team.",
    ])
    assert result is True, (
        "Expected True when ticket ID is in the second response. "
        f"Got: {result}"
    )

    # Case 4: empty list
    result = exercise_9_2_answer([])
    assert result is False, (
        f"Expected False for empty response list. Got: {result}"
    )

    print("[PASS] Exercise 9.2 passed.")


test_exercise_9_2()

## Solutions

In [ ]:
# SOLUTION 9.1: Multi-turn Create Ticket test case

create_ticket_multi_turn_solution = AgentTestCase(
    name="Full Create Ticket conversation",
    messages=[
        "I need to submit a ticket",  # Trigger the topic
        "Software",                   # Category selection
        "Outlook keeps crashing when I open attachments",  # Description
        "High",                       # Priority
        "Yes, submit",                # Confirmation
    ],
    expected_phrases=["INC-", "Software Team"],
    forbidden_phrases=["Sorry, I didn't understand", "I couldn't help"],
)


# SOLUTION 9.2: Ticket ID validator

def validate_ticket_id_in_response_solution(responses: list[str]) -> bool:
    """
    Check whether any response string contains a formatted ticket ID.
    """
    pattern = r"INC-\d+"
    combined = " ".join(responses)
    return bool(re.search(pattern, combined))

In [ ]:
section_divider("Summary")

## Summary

### Key Takeaways

1. **Power Platform API** — the `/bots` endpoint lets you list and inspect agents programmatically without opening Copilot Studio.
2. **Direct Line API** — a REST protocol that lets Python (or any HTTP client) conduct full multi-turn conversations with a Copilot Studio agent.
3. **DirectLineSession** — open a conversation, send messages, receive responses, track the watermark for incremental polling.
4. **Test harness** — define expected and forbidden phrases per test case; run all cases in sequence; report pass/fail with enough detail to act on failures.
5. **Transcript parsing** — activities contain `suggestedActions` (quick replies) that multi-turn tests must select by sending the action's `value` as the next message.

### What's Next

With the API harness working, add it to a GitHub Actions workflow:
- Trigger on pull requests that modify agent flows
- Run the full test suite against a staging environment
- Fail the PR if any test case fails

This turns your test harness into a regression gate that prevents broken agent deployments.

### Additional Resources

- [Direct Line API 3.0 reference](https://learn.microsoft.com/en-us/azure/bot-service/rest-api/bot-framework-rest-direct-line-3-0-concepts)
- [Power Platform API authentication](https://learn.microsoft.com/en-us/power-platform/admin/programmability-authentication)
- [Bot Framework Activity schema](https://learn.microsoft.com/en-us/azure/bot-service/rest-api/bot-framework-rest-connector-api-reference)

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])